# 🔗 ImageBind — CSC14005 Final Project
**Topic 17 | Nhóm 2 người | Focused type**

| Tuần | Nội dung |
|------|----------|
| Tuần 1 | Setup môi trường · Clone repo · Chạy demo · Chuẩn bị data |
| Tuần 2 | **Training loop (FIXED)** · Freeze backbone · Checkpoint · Evaluate · Metric |
| Tuần 3 | Benchmark hyperparams · Full evaluate · Vẽ biểu đồ |

> **Chạy trên Kaggle**: Accelerator → GPU T4 · Internet → ON

---
# 📗 TUẦN 1 — Setup & Chuẩn bị

## 1.1 Cài đặt môi trường

In [8]:
import os

if not os.path.exists('ImageBind'):
    !git clone https://github.com/facebookresearch/ImageBind.git
    print('Cloned!')
else:
    print('Repo da ton tai, bo qua.')

%cd ImageBind

!pip install -q .
!pip install -q timm ftfy regex einops fvcore tqdm matplotlib seaborn pandas
print('\n=== Cai dat xong ===')

Cloning into 'ImageBind'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 187 (delta 84), reused 48 (delta 48), pack-reused 67 (from 2)
Receiving objects: 100% (187/187), 2.65 MiB | 15.52 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Cloned!
/kaggle/working/ImageBind/ImageBind
  Preparing metadata (setup.py) ... done

=== Cai dat xong ===


In [9]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch

print('=== Kiem tra moi truong ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nSu dung device  : {DEVICE}')

=== Kiem tra moi truong ===
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB

Su dung device  : cuda


## 1.2 Tải pretrained weights

In [10]:
import os

os.makedirs('.checkpoints', exist_ok=True)
WEIGHT_PATH = '/kaggle/input/datasets/hongngctng/checkpoint1/imagebind_huge.pth'

if not os.path.exists(WEIGHT_PATH):
    print('Dang tai pretrained weights (~4.5GB)...')
    !wget -q --show-progress -O {WEIGHT_PATH} \
        https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth
    print('Tai xong!')
else:
    size_mb = os.path.getsize(WEIGHT_PATH) / 1e6
    print(f'Weights da co san: {size_mb:.0f} MB')

Weights da co san: 4804 MB


## 1.3 Load model & Smoke test

In [11]:
from imagebind import data as ibdata
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

print('Dang load model...')
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(DEVICE)
print('Load model thanh cong!')

total_params = sum(p.numel() for p in model.parameters())
print(f'Tong params: {total_params / 1e6:.1f}M')
print('(Sau khi freeze backbone o Tuan 2, chi con ~vài MB trainable)')

Dang load model...


100%|██████████| 4.47G/4.47G [00:13<00:00, 353MB/s]


Load model thanh cong!
Tong params: 1200.8M
(Sau khi freeze backbone o Tuan 2, chi con ~vài MB trainable)


In [12]:
import torch

texts = ['A dog is barking', 'Fire crackling sound',
         'Ocean waves crashing', 'Rain falling on roof']

inputs = {ModalityType.TEXT: ibdata.load_and_transform_text(texts, DEVICE)}

with torch.no_grad():
    embeddings = model(inputs)

emb = embeddings[ModalityType.TEXT]
print(f'Text embedding shape: {emb.shape}')  # [4, 1024]
print(f'Embedding norm      : {emb.norm(dim=-1)}')  # ~1.0
print('Smoke test OK!')

Text embedding shape: torch.Size([4, 1024])
Embedding norm      : tensor([100.0000, 100.0000, 100.0000, 100.0000], device='cuda:0')
Smoke test OK!


---
# 📘 TUẦN 2 — Training Loop (FIXED)

**Các fix so với lần trước:**
| # | Vấn đề cũ | Fix |
|---|---|---|
| 1 | Train toàn bộ 1.2B params | ✅ Freeze backbone, chỉ train head |
| 2 | 2 epoch → acc=5% | ✅ 30 epoch với OneCycleLR |
| 3 | Không resume được | ✅ Load checkpoint đúng cách |
| 4 | Không có label smoothing | ✅ CrossEntropyLoss(label_smoothing=0.1) |
| 5 | Không clip gradient | ✅ clip_grad_norm_(max=1.0) |

## 2.1 Dataset — ESC-50

In [15]:
import os, glob, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

# ── Đường dẫn ESC-50 trên Kaggle ──
ESC50_ROOT  = '/kaggle/input/datasets/hongngctng/dataesc50/ESC-50-master'
AUDIO_DIR   = os.path.join(ESC50_ROOT, 'audio')
META_PATH   = os.path.join(ESC50_ROOT, 'meta', 'esc50.csv')

# ── Mel-spectrogram config (khớp với ImageBind) ──
# ── Audio config khớp với ImageBind ──
SAMPLE_RATE  = 16_000
N_MELS       = 128
N_FFT        = 400       # ← đổi từ 1024
HOP_LENGTH   = 160       # ← đổi từ 512
DURATION_SEC = 2         # ← đổi từ 5 (ImageBind dùng 2 giây)
TARGET_LEN   = SAMPLE_RATE * DURATION_SEC  # 32000 samples
TARGET_FRAMES = 204      # ← số frame cố định ImageBind expect

mel_transform = T.MelSpectrogram(
    sample_rate = SAMPLE_RATE,
    n_fft       = N_FFT,
    hop_length  = HOP_LENGTH,
    n_mels      = N_MELS,
)   # CPU, không .to(DEVICE)

class ESC50Dataset(Dataset):
    """ESC-50 dataset trả về (mel_spec, label)."""
    def __init__(self, meta_df, audio_dir, augment=False):
        self.meta      = meta_df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.augment   = augment

        classes        = sorted(meta_df['category'].unique())
        self.class2idx = {c: i for i, c in enumerate(classes)}
        self.idx2class = {i: c for c, i in self.class2idx.items()}
        self.num_classes = len(classes)

    def __len__(self):
        return len(self.meta)

    def _load_wav(self, path):
        wav, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            wav = T.Resample(sr, SAMPLE_RATE)(wav)
        wav = wav.mean(dim=0)           # stereo → mono
        if wav.shape[0] < TARGET_LEN:
            wav = wav.repeat(int(np.ceil(TARGET_LEN / wav.shape[0])))[:TARGET_LEN]
        else:
            wav = wav[:TARGET_LEN]
        return wav                      # (80000,)

    def _augment(self, wav):
        # Time shift
        shift = random.randint(-8000, 8000)
        wav   = torch.roll(wav, shift)
        # Gaussian noise
        wav   = wav + 0.005 * torch.randn_like(wav)
        return wav

    def __getitem__(self, idx):
        row      = self.meta.iloc[idx]
        wav_path = os.path.join(self.audio_dir, row['filename'])
        wav      = self._load_wav(wav_path)
        if self.augment:
            wav  = self._augment(wav)

        mel = mel_transform(wav)
        mel      = (mel + 1e-9).log2()                  # log scale
        mel      = (mel - mel.mean()) / (mel.std() + 1e-6)  # normalize
        label    = self.class2idx[row['category']]
        return mel, label


# ── Train/Val/Test split (fold 5 = test) ──
meta = pd.read_csv(META_PATH)
test_meta  = meta[meta['fold'] == 5]
train_meta = meta[meta['fold'].isin([1, 2, 3])]
val_meta   = meta[meta['fold'] == 4]

train_dataset = ESC50Dataset(train_meta, AUDIO_DIR, augment=True)
val_dataset   = ESC50Dataset(val_meta,   AUDIO_DIR, augment=False)
test_dataset  = ESC50Dataset(test_meta,  AUDIO_DIR, augment=False)

NUM_CLASSES = train_dataset.num_classes

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Num classes: {NUM_CLASSES}')
print('Classes:', list(train_dataset.class2idx.keys())[:5], '...')

Train: 1200 | Val: 400 | Test: 400
Num classes: 50
Classes: ['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn'] ...


In [16]:
BATCH_SIZE = 32   # Giảm xuống 16 nếu bị OOM
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

# Kiểm tra shape
mel_b, lbl_b = next(iter(train_loader))
print(f'Mel batch shape : {mel_b.shape}')   # (B, 128, T)
print(f'Label batch     : {lbl_b[:5]}')

Train batches: 37 | Val batches: 13
Mel batch shape : torch.Size([32, 128, 157])
Label batch     : tensor([43, 21, 34, 30, 34])


## 2.2 Freeze backbone — FIX #1

> **Mục đích**: Giữ nguyên pretrained weights của ImageBind, chỉ train audio head (~vài MB thay vì 1.2GB). VRAM từ ~15GB → ~4GB.

In [17]:
def setup_freeze(backbone, mode: str = 'head'):
    """
    mode='head'  → chỉ train audio projection head  (nhanh, ít VRAM nhất)
    mode='top2'  → thêm 2 transformer block cuối    (chậm hơn, acc cao hơn)
    mode='full'  → train toàn bộ                    (KHÔNG khuyến nghị T4)
    """
    # Đóng tất cả trước
    for p in backbone.parameters():
        p.requires_grad = False

    if mode in ('head', 'top2'):
        # Mở audio head + postprocessor
        for p in backbone.modality_heads[ModalityType.AUDIO].parameters():
            p.requires_grad = True
        for p in backbone.modality_postprocessors[ModalityType.AUDIO].parameters():
            p.requires_grad = True

    if mode == 'top2':
        # Thêm 2 transformer block cuối của audio trunk
        audio_trunk = backbone.modality_trunks[ModalityType.AUDIO]
        trunk_children = list(audio_trunk.children())
        for block in trunk_children[-2:]:
            for p in block.parameters():
                p.requires_grad = True

    elif mode == 'full':
        for p in backbone.parameters():
            p.requires_grad = True

    total     = sum(p.numel() for p in backbone.parameters())
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f'[Freeze] mode="{mode}"')
    print(f'  Trainable : {trainable/1e6:.2f}M / {total/1e6:.1f}M params')
    return backbone


FREEZE_MODE = 'head'   # ← thay 'top2' nếu muốn acc cao hơn
model = setup_freeze(model, mode=FREEZE_MODE)

[Freeze] mode="head"
  Trainable : 0.79M / 1200.8M params


## 2.3 AudioClassifier — head gắn lên embedding

In [18]:
import torch.nn as nn
import torch.nn.functional as F


class AudioClassifier(nn.Module):
    """Backbone ImageBind + linear classifier head."""
    def __init__(self, backbone, embed_dim: int = 1024, num_classes: int = 50):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, mel):
        """mel: (B, 1, 128, T)"""
        emb = self.backbone({ModalityType.AUDIO: mel})[ModalityType.AUDIO]
        return self.classifier(emb)   # (B, num_classes)


train_model = AudioClassifier(
    backbone    = model,
    embed_dim   = 1024,
    num_classes = NUM_CLASSES,
).to(DEVICE)

print('AudioClassifier sẵn sàng!')

# Kiểm tra forward pass
with torch.no_grad():
    dummy = torch.randn(2, 1, 128, 157).to(DEVICE)
    out   = train_model(dummy)
print(f'Output shape: {out.shape}')  # (2, 50)

AudioClassifier sẵn sàng!


AssertionError: Interpolation of pos embed not supported for non-square layouts

## 2.4 Optimizer, Scheduler, Loss — FIX #2 #3

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

NUM_EPOCHS = 30
LR_MAX     = 1e-4

# Optimizer — chỉ update params đang requires_grad
trainable_params = filter(lambda p: p.requires_grad, train_model.parameters())
optimizer = torch.optim.AdamW(
    trainable_params,
    lr           = LR_MAX,
    weight_decay = 1e-4,
)

# OneCycleLR — warmup 10% epoch đầu, cosine decay sau đó
scheduler = OneCycleLR(
    optimizer,
    max_lr          = LR_MAX,
    epochs          = NUM_EPOCHS,
    steps_per_epoch = len(train_loader),
    pct_start       = 0.1,
    anneal_strategy = 'cos',
)

# CrossEntropyLoss + label smoothing — FIX #3
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Optimizer : AdamW | LR_max={LR_MAX}')
print(f'Scheduler : OneCycleLR | {NUM_EPOCHS} epochs')
print(f'Loss      : CrossEntropyLoss(label_smoothing=0.1)')

## 2.5 Checkpoint — Resume & Save — FIX #4

In [ ]:
import os, json

CKPT_DIR  = 'checkpoints'
RESUME    = os.path.join(CKPT_DIR, 'best_model.pth')  # None = train từ đầu
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)


def save_checkpoint(model, optimizer, scheduler, epoch, best_acc, history, tag='best'):
    path = os.path.join(CKPT_DIR, f'ckpt_{tag}.pth')
    torch.save({
        'epoch'                 : epoch,
        'best_acc'              : best_acc,
        'backbone_state_dict'   : model.backbone.state_dict(),
        'classifier_state_dict' : model.classifier.state_dict(),
        'optimizer_state_dict'  : optimizer.state_dict(),
        'scheduler_state_dict'  : scheduler.state_dict() if scheduler else None,
        'history'               : history,
    }, path)
    print(f'  → Saved: {path}')


def load_checkpoint(model, optimizer, scheduler, path):
    if path and os.path.exists(path):
        ckpt = torch.load(path, map_location=DEVICE)

        # Load backbone
        missing, unexpected = model.backbone.load_state_dict(
            ckpt.get('backbone_state_dict', ckpt.get('model_state_dict', {})),
            strict=False
        )
        if missing:
            print(f'  [WARN] Missing keys: {len(missing)}')

        # Load classifier head
        if 'classifier_state_dict' in ckpt:
            model.classifier.load_state_dict(ckpt['classifier_state_dict'])

        # Load optimizer & scheduler
        if optimizer and 'optimizer_state_dict' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            except Exception as e:
                print(f'  [WARN] Optimizer load failed (reset): {e}')

        if scheduler and ckpt.get('scheduler_state_dict'):
            try:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            except Exception as e:
                print(f'  [WARN] Scheduler load failed (reset): {e}')

        start_epoch = ckpt.get('epoch', 0) + 1
        best_acc    = ckpt.get('best_acc', 0.0)
        history     = ckpt.get('history', [])
        print(f'[Resume] epoch={start_epoch} | best_acc={best_acc:.2f}%')
        return start_epoch, best_acc, history
    else:
        print('[Start] Khong co checkpoint — train tu dau')
        return 0, 0.0, []


start_epoch, best_acc, history = load_checkpoint(train_model, optimizer, scheduler, RESUME)

## 2.6 Training Loop chính — FIX #5 (gradient clipping)

In [ ]:
import time
from tqdm import tqdm


def run_epoch(model, loader, criterion, optimizer, scheduler, device, is_train: bool):
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc='Train' if is_train else 'Val  ', leave=False)
    for mel_batch, label_batch in pbar:
        mel_input = mel_batch.unsqueeze(1).to(device)   # (B, 1, 128, T)
        labels    = label_batch.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(mel_input)                    # (B, num_classes)
            loss   = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            # FIX #5 — Gradient clipping
            nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0,
            )
            optimizer.step()
            if scheduler:
                scheduler.step()

        preds       = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)

        pbar.set_postfix(
            loss=f'{loss.item():.4f}',
            acc=f'{correct/total*100:.1f}%',
        )

    return total_loss / total, correct / total * 100


# ─────────── Main loop ───────────
print(f'{'='*58}')
print(f'  Training | freeze={FREEZE_MODE} | epochs={NUM_EPOCHS} | device={DEVICE}')
print(f'{'='*58}\n')

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = run_epoch(
        train_model, train_loader, criterion, optimizer, scheduler, DEVICE, is_train=True,
    )
    val_loss, val_acc = run_epoch(
        train_model, val_loader, criterion, None, None, DEVICE, is_train=False,
    )

    elapsed    = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    log = {
        'epoch'     : epoch,
        'train_loss': round(train_loss, 4),
        'train_acc' : round(train_acc, 2),
        'val_loss'  : round(val_loss, 4),
        'val_acc'   : round(val_acc, 2),
        'lr'        : round(current_lr, 8),
        'time_s'    : round(elapsed, 1),
    }
    history.append(log)

    print(
        f'Epoch [{epoch+1:02d}/{NUM_EPOCHS}] '
        f'| Train loss={train_loss:.4f} acc={train_acc:.1f}% '
        f'| Val loss={val_loss:.4f} acc={val_acc:.1f}% '
        f'| LR={current_lr:.2e} | {elapsed:.0f}s'
    )

    # Lưu best checkpoint
    if val_acc > best_acc:
        best_acc = val_acc
        save_checkpoint(train_model, optimizer, scheduler, epoch, best_acc, history, tag='best')
        print(f'  ★ New best val acc: {best_acc:.2f}%')

    # Lưu định kỳ mỗi 5 epoch
    if (epoch + 1) % 5 == 0:
        save_checkpoint(train_model, optimizer, scheduler, epoch, best_acc, history, tag=f'ep{epoch+1:02d}')

# Lưu history ra JSON
with open('results/train_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n✅ Training xong! Best val acc: {best_acc:.2f}%')
print('Checkpoint luu tai: checkpoints/')

## 2.7 Learning curve

In [ ]:
import matplotlib.pyplot as plt

epochs_x   = [h['epoch'] + 1 for h in history]
train_acc  = [h['train_acc']  for h in history]
val_acc    = [h['val_acc']    for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss']   for h in history]
lr_hist    = [h['lr']         for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'ImageBind Fine-tuning — freeze={FREEZE_MODE}', fontsize=12, fontweight='bold')

# Accuracy
axes[0].plot(epochs_x, train_acc, label='Train', marker='o', markersize=3)
axes[0].plot(epochs_x, val_acc,   label='Val',   marker='s', markersize=3)
axes[0].axhline(best_acc, color='red', linestyle='--', alpha=0.5, label=f'Best={best_acc:.1f}%')
axes[0].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_x, train_loss, label='Train', marker='o', markersize=3)
axes[1].plot(epochs_x, val_loss,   label='Val',   marker='s', markersize=3)
axes[1].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

# LR schedule
axes[2].plot(epochs_x, lr_hist, color='darkorange', marker='.', markersize=3)
axes[2].set(xlabel='Epoch', ylabel='Learning Rate', title='LR (OneCycleLR)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/learning_curve.png', dpi=120)
plt.show()
print('Luu: results/learning_curve.png')

## 2.8 Evaluate trên Test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Load best checkpoint trước khi evaluate
best_ckpt = torch.load(os.path.join(CKPT_DIR, 'ckpt_best.pth'), map_location=DEVICE)
train_model.backbone.load_state_dict(best_ckpt['backbone_state_dict'], strict=False)
train_model.classifier.load_state_dict(best_ckpt['classifier_state_dict'])
print(f'Loaded best checkpoint | epoch={best_ckpt["epoch"]+1} | best_acc={best_ckpt["best_acc"]:.2f}%')

train_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for mel_batch, label_batch in tqdm(test_loader, desc='Test eval'):
        mel_input = mel_batch.unsqueeze(1).to(DEVICE)
        logits    = train_model(mel_input)
        preds     = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(label_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean() * 100

print(f'\n=== Test Accuracy: {test_acc:.2f}% ===\n')

class_names = [test_dataset.idx2class[i] for i in range(NUM_CLASSES)]
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.3,
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(f'Confusion Matrix — Test Acc={test_acc:.1f}%', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=120)
plt.show()
print('Luu: results/confusion_matrix.png')

In [ ]:
# Top-5 accuracy
train_model.eval()
correct_top5, total_top5 = 0, 0

with torch.no_grad():
    for mel_batch, label_batch in test_loader:
        mel_input = mel_batch.unsqueeze(1).to(DEVICE)
        logits    = train_model(mel_input)
        top5      = logits.topk(5, dim=-1).indices.cpu()
        labels    = label_batch.unsqueeze(1).expand_as(top5)
        correct_top5 += (top5 == labels).any(dim=1).sum().item()
        total_top5   += label_batch.size(0)

top5_acc = correct_top5 / total_top5 * 100
print(f'Top-1 Accuracy : {test_acc:.2f}%')
print(f'Top-5 Accuracy : {top5_acc:.2f}%')

# Lưu kết quả
results_summary = {
    'freeze_mode' : FREEZE_MODE,
    'num_epochs'  : NUM_EPOCHS,
    'best_val_acc': round(best_acc, 2),
    'test_top1'   : round(test_acc, 2),
    'test_top5'   : round(top5_acc, 2),
    'history'     : history,
}
with open('results/final_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print('Luu: results/final_results.json')

---
# 📙 TUẦN 3 — Benchmark & Cross-modal Demo

## 3.1 Zero-shot Text → Audio Retrieval

In [ ]:
@torch.no_grad()
def demo_text_to_audio(model, query_texts, test_dataset, test_loader, device, top_k=5):
    model.eval()
    # Text embeddings
    text_in  = {ModalityType.TEXT: ibdata.load_and_transform_text(query_texts, device)}
    text_emb = model.backbone(text_in)[ModalityType.TEXT]          # (Q, 1024)
    text_emb = F.normalize(text_emb, dim=-1).cpu()

    # Audio embeddings
    audio_embs, audio_labels = [], []
    for mel_batch, label_batch in test_loader:
        mel_in = mel_batch.unsqueeze(1).to(device)
        a_emb  = model.backbone({ModalityType.AUDIO: mel_in})[ModalityType.AUDIO]
        a_emb  = F.normalize(a_emb, dim=-1).cpu()
        audio_embs.append(a_emb)
        audio_labels.extend(label_batch.numpy())
    audio_embs = torch.cat(audio_embs, dim=0)   # (N, 1024)

    # Similarity
    sim = text_emb @ audio_embs.T              # (Q, N)
    results = []
    for q_idx, q_text in enumerate(query_texts):
        topk_idx    = sim[q_idx].topk(top_k).indices.numpy()
        topk_labels = [test_dataset.idx2class[audio_labels[i]] for i in topk_idx]
        topk_scores = sim[q_idx][topk_idx].numpy()
        results.append({'query': q_text, 'top_classes': topk_labels, 'top_scores': topk_scores.tolist()})
    return results


TEXT_QUERIES = [
    'sound of a dog barking outside',
    'rain falling on a rooftop',
    'crackling fire in a fireplace',
    'ocean waves on a beach',
    'a baby crying loudly',
]

print('=== Demo: Text -> Audio Retrieval (Top-5) ===')
t2a_results = demo_text_to_audio(
    model=train_model, query_texts=TEXT_QUERIES,
    test_dataset=test_dataset, test_loader=test_loader,
    device=DEVICE, top_k=5,
)

for r in t2a_results:
    print(f"\nQuery: '{r['query']}'")
    for rank, (cls, sc) in enumerate(zip(r['top_classes'], r['top_scores']), 1):
        print(f'  #{rank}: {cls:<25s} (score={sc:.4f})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_queries = len(t2a_results)
fig, axes = plt.subplots(1, n_queries, figsize=(4 * n_queries, 5))
fig.suptitle('Text → Audio Retrieval (Top-5 per Query)', fontsize=12, fontweight='bold')

for ax, res in zip(axes, t2a_results):
    classes = res['top_classes']
    scores  = res['top_scores']
    ax.barh(range(len(classes))[::-1], scores, color='steelblue', alpha=0.85)
    ax.set_yticks(range(len(classes))[::-1])
    ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel('Cosine Score')
    q_short = res['query'][:35] + '...' if len(res['query']) > 35 else res['query']
    ax.set_title(q_short, fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/demo_text_to_audio.png', dpi=120)
plt.show()
print('Luu: results/demo_text_to_audio.png')

## 3.2 Recall@K — Cross-modal

In [ ]:
@torch.no_grad()
def compute_recall_at_k(model, test_dataset, test_loader, device, ks=(1, 5, 10)):
    model.eval()
    class_names  = [test_dataset.idx2class[i] for i in range(test_dataset.num_classes)]
    text_in      = {ModalityType.TEXT: ibdata.load_and_transform_text(class_names, device)}
    text_embs    = F.normalize(model.backbone(text_in)[ModalityType.TEXT], dim=-1).cpu()  # (C, 1024)

    audio_embs, audio_labels = [], []
    for mel_batch, label_batch in test_loader:
        mel_in = mel_batch.unsqueeze(1).to(device)
        a_emb  = F.normalize(model.backbone({ModalityType.AUDIO: mel_in})[ModalityType.AUDIO], dim=-1).cpu()
        audio_embs.append(a_emb)
        audio_labels.extend(label_batch.numpy())
    audio_embs   = torch.cat(audio_embs, dim=0)  # (N, 1024)
    audio_labels = np.array(audio_labels)

    # Audio → Text
    sim_a2t      = audio_embs @ text_embs.T      # (N, C)
    a2t_recall   = {}
    for k in ks:
        topk       = sim_a2t.topk(k, dim=-1).indices.numpy()
        hits       = np.array([audio_labels[i] in topk[i] for i in range(len(audio_labels))])
        a2t_recall[f'R@{k}'] = round(hits.mean() * 100, 2)

    # Text → Audio
    sim_t2a    = text_embs @ audio_embs.T        # (C, N)
    t2a_recall = {}
    for k in ks:
        topk   = sim_t2a.topk(k, dim=-1).indices.numpy()
        hits   = []
        for c_idx in range(len(class_names)):
            true_audio_idx = np.where(audio_labels == c_idx)[0]
            hit = any(ai in topk[c_idx] for ai in true_audio_idx)
            hits.append(hit)
        t2a_recall[f'R@{k}'] = round(np.mean(hits) * 100, 2)

    return a2t_recall, t2a_recall


a2t_recall, t2a_recall = compute_recall_at_k(train_model, test_dataset, test_loader, DEVICE)
print('Audio → Text Recall@K:', a2t_recall)
print('Text → Audio Recall@K:', t2a_recall)

with open('results/retrieval_recall.json', 'w') as f:
    json.dump({'audio_to_text': a2t_recall, 'text_to_audio': t2a_recall}, f, indent=2)

## 3.3 Tổng hợp kết quả

In [ ]:
import os, json
import pandas as pd

print('=' * 60)
print('  TONG HOP KET QUA — ImageBind CSC14005')
print('=' * 60)

with open('results/final_results.json') as f:
    res = json.load(f)

print(f'\n[Config]')
print(f'  Freeze mode : {res["freeze_mode"]}')
print(f'  Epochs      : {res["num_epochs"]}')

print(f'\n[Accuracy]')
print(f'  Best Val Top-1 : {res["best_val_acc"]:.2f}%')
print(f'  Test Top-1     : {res["test_top1"]:.2f}%')
print(f'  Test Top-5     : {res["test_top5"]:.2f}%')

with open('results/retrieval_recall.json') as f:
    ret = json.load(f)
print(f'\n[Cross-modal Retrieval]')
print(pd.DataFrame({'Audio→Text': ret['audio_to_text'], 'Text→Audio': ret['text_to_audio']}).to_string())

print(f'\n[Output files]')
for fn in sorted(os.listdir('results')):
    fp = os.path.join('results', fn)
    print(f'  {fn:45s} ({os.path.getsize(fp):,} bytes)')